# NYC Yellow Taxi 2019: PySpark Data Processing Pipeline

This notebook builds a PySpark pipeline on one full year of NYC Yellow Taxi trip records (2019, about 1.2 GB of Parquet, tens of millions of trips) plus the TLC Taxi Zone Lookup table.

**Sections**
1. Setup: start a local Spark session
2. Load the data and standardize the 12 monthly files into one partitioned Parquet dataset
3. Transformations: filters, `withColumn`, a broadcast join and `groupBy` aggregations
4. SQL queries
5. Optimization: execution plans, filter pushdown, partition pruning, broadcast vs shuffle join
6. Caching: repeated actions with and without `.cache()`
7. Actions vs transformations (lazy vs eager)
8. Write results to Parquet and CSV
9. Optional: MLlib linear regression to predict the fare

**Data Location:** 

The raw data is kept outside this repository in `~/spark-data/a12` (see the README for the download commands), because the files are far too large for GitHub.

## 1. Setup

We start Spark in **local mode**. `local[*]` means Spark uses every CPU core on this computer.

In [1]:
import os
import glob
import time
from functools import reduce

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_DIR = os.getcwd()
DATA_DIR = os.path.expanduser("~/spark-data/a12")
RAW_DIR = os.path.join(DATA_DIR, "trips")
ZONES_PATH = os.path.join(DATA_DIR, "taxi_zone_lookup.csv")
CLEAN_DIR = os.path.join(DATA_DIR, "clean_trips")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

spark = (
    SparkSession.builder.appName("NYC Taxi 2019 Pipeline")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.hadoop.parquet.hadoop.vectored.io.enabled", "false")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Cores used:", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/20 01:58:55 WARN Utils: Your hostname, Muhammads-MacBook-Pro-400.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.4 instead (on interface en0)
26/11/20 01:58:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 01:58:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.3.0
Cores used: 10
Spark UI: http://172.20.10.4:4040


In [2]:
def timed(label, action):
    """Run a function, print how long it took, and return (result, seconds)."""
    start = time.perf_counter()
    result = action()
    seconds = time.perf_counter() - start
    print(f"{label}: {seconds:.2f} seconds")
    return result, seconds

## 2. Load the data

### 2.1 Raw files

The TLC publishes one Parquet file per month. Before reading them all together, we check that every file stores each column with the same data type. Reading a Parquet file's schema only reads the small footer of the file, so this is fast.

In [3]:
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "yellow_tripdata_2019-*.parquet")))
print("Monthly files found:", len(raw_files))
print(
    "Total size on disk: %.2f GB"
    % (sum(os.path.getsize(f) for f in raw_files) / 1024**3)
)

schema_by_file = {}
for path in raw_files:
    month_label = (
        os.path.basename(path).replace("yellow_tripdata_", "").replace(".parquet", "")
    )
    schema = spark.read.parquet(path).schema
    schema_by_file[month_label] = {
        field.name: field.dataType.simpleString() for field in schema.fields
    }

schema_table = pd.DataFrame(schema_by_file)
columns_with_mixed_types = schema_table[schema_table.nunique(axis=1, dropna=False) > 1]
print("Columns whose type or presence differs between months:")
columns_with_mixed_types

Monthly files found: 12
Total size on disk: 1.16 GB
Columns whose type or presence differs between months:


,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12


### 2.2 Standardize into one clean, partitioned dataset

Monthly files can store the same column with different types (for example an integer in one month and a decimal number in another), and some later columns do not exist in every file. Reading such files together can fail.

So we read each month on its own, **cast every column to one agreed type**, add a `file_month` column, and write everything once into a single Parquet dataset **partitioned by `file_month`**. Partitioning means Spark stores each month in its own folder (`file_month=1`, `file_month=2`, and so on). A later query that asks for one month can then skip the other 11 folders completely.

This step is skipped automatically if the clean dataset already exists.

In [4]:
TARGET_TYPES = {
    "VendorID": "long",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "RatecodeID": "double",
    "store_and_fwd_flag": "string",
    "PULocationID": "long",
    "DOLocationID": "long",
    "payment_type": "long",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
}


def read_month(path):
    month = int(os.path.basename(path)[-10:-8])
    df = spark.read.parquet(path)
    columns = [
        (F.col(name) if name in df.columns else F.lit(None)).cast(dtype).alias(name)
        for name, dtype in TARGET_TYPES.items()
    ]
    return df.select(*columns).withColumn("file_month", F.lit(month))


if os.path.exists(os.path.join(CLEAN_DIR, "_SUCCESS")):
    print("Clean dataset already exists, skipping the rewrite:", CLEAN_DIR)
else:
    standardized = reduce(
        lambda a, b: a.unionByName(b), [read_month(p) for p in raw_files]
    )
    timed(
        "Write standardized dataset (partitioned by file_month)",
        lambda: standardized.write.mode("overwrite")
        .partitionBy("file_month")
        .parquet(CLEAN_DIR),
    )

Clean dataset already exists, skipping the rewrite: /Users/mbaqir/spark-data/a12/clean_trips


### 2.3 Load the clean trips and the zone lookup table

In [5]:
trips = spark.read.parquet(CLEAN_DIR)

zones = spark.read.csv(ZONES_PATH, header=True, inferSchema=True).withColumn(
    "LocationID", F.col("LocationID").cast("long")
)

trips.printSchema()
total_rows, _ = timed("Count all trip rows", lambda: trips.count())
print(f"Total trips in 2019 files: {total_rows:,}")
print("Input partitions (file splits) Spark will read:", trips.rdd.getNumPartitions())
zones.show(5, truncate=False)

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- file_month: integer (nullable = true)

Count all trip rows: 0.34 seconds
Total trips in 2019 files: 84,598,444
Input partitions (file splits) Spark will read: 15
+----------+---

In [6]:
trips.show(3, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|file_month|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+----------+
|1       |2019-05-01 00:14:50 |2019-05-01 00:16:48  |1.0            |0.0          |1.0       |N                 |145         |145         |2           |3.0        |0.5  |0.5    |0.0       |0

## 3. Transformations

The pipeline below follows the "filter early" rule:

1. **Column pruning:** `select` only the 15 columns we use, so Spark never reads the other columns from disk.
2. **Filters first:** remove bad rows before any expensive work.
   * Filter 1: keep only pickups inside calendar year 2019 (some files contain stray 2018 or 2020 dates).
   * Filter 2: keep realistic distances (more than 0 and at most 100 miles).
   * Filter 3: keep positive fares and totals.
   * Filter 4: keep 1 to 6 passengers.
3. **`withColumn` features:** trip length in minutes, average speed, tip percent, hour, month, weekday, a weekend flag and a readable payment method.
4. Two more quality filters that need the new columns: 1 to 180 minutes, and at most 80 mph.

Nothing runs yet. These are all **transformations**, so Spark only builds a plan. The work happens later, when an **action** such as `count()` or `write` asks for a result.

In [7]:
TRIP_COLUMNS = [
    "file_month",
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "congestion_surcharge",
    "RatecodeID",
]

YEAR_START = F.lit("2019-01-01 00:00:00").cast("timestamp")
YEAR_END = F.lit("2020-01-01 00:00:00").cast("timestamp")

trips_clean = (
    trips.select(*TRIP_COLUMNS)  # column pruning
    .filter(
        (F.col("tpep_pickup_datetime") >= YEAR_START)
        & (F.col("tpep_pickup_datetime") < YEAR_END)
    )  # filter 1
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100))  # filter 2
    .filter((F.col("fare_amount") > 0) & (F.col("total_amount") > 0))  # filter 3
    .filter(F.col("passenger_count").between(1, 6))  # filter 4
)

trips_features = (
    trips_clean.withColumn(
        "trip_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        )
        / 60.0,
    )
    .filter(F.col("trip_minutes").between(1, 180))
    .withColumn(
        "avg_speed_mph", F.col("trip_distance") / (F.col("trip_minutes") / 60.0)
    )
    .filter(F.col("avg_speed_mph") <= 80)
    .withColumn("tip_pct", F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "E"))
    .withColumn("is_weekend", F.dayofweek("tpep_pickup_datetime").isin(1, 7))
    .withColumn(
        "payment_method",
        F.when(F.col("payment_type") == 1, "Credit card")
        .when(F.col("payment_type") == 2, "Cash")
        .when(F.col("payment_type") == 3, "No charge")
        .when(F.col("payment_type") == 4, "Dispute")
        .otherwise("Other"),
    )
)

### 3.1 Join with the zone lookup table

The trip table only has zone numbers (`PULocationID`, `DOLocationID`). We join it to the zone table twice, once for the pickup zone and once for the drop-off zone, to get borough and neighborhood names.

The zone table has only about 265 rows, so we wrap it in `F.broadcast(...)`. Spark then sends a full copy of the small table to every worker, and each worker matches its own trips locally. The big trip table never has to be shuffled (moved between workers) for the join.

In [8]:
pickup_zones = zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
)
dropoff_zones = zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone"),
)

trips_enriched = trips_features.join(
    F.broadcast(pickup_zones), on="PULocationID", how="left"
).join(F.broadcast(dropoff_zones), on="DOLocationID", how="left")

clean_rows, _ = timed(
    "Count rows after filters and joins", lambda: trips_enriched.count()
)
print(f"Rows kept: {clean_rows:,} of {total_rows:,} ({clean_rows / total_rows:.1%})")
trips_enriched.select(
    "tpep_pickup_datetime",
    "pickup_zone",
    "dropoff_zone",
    "trip_distance",
    "trip_minutes",
    "avg_speed_mph",
    "fare_amount",
    "tip_pct",
    "payment_method",
).show(5, truncate=False)

Count rows after filters and joins: 1.94 seconds
Rows kept: 81,228,290 of 84,598,444 (96.0%)
+--------------------+------------------------------+------------------------------+-------------+------------------+------------------+-----------+-------+--------------+
|tpep_pickup_datetime|pickup_zone                   |dropoff_zone                  |trip_distance|trip_minutes      |avg_speed_mph     |fare_amount|tip_pct|payment_method|
+--------------------+------------------------------+------------------------------+-------------+------------------+------------------+-----------+-------+--------------+
|2019-05-31 07:00:00 |Yorkville East                |Midtown East                  |2.2          |7.933333333333334 |16.638655462184875|9.5        |15.79  |Credit card   |
|2019-05-01 00:35:54 |Long Island City/Hunters Point|Long Island City/Hunters Point|1.5          |1.55              |58.064516129032256|3.0        |0.0    |Cash          |
|2019-05-31 07:41:02 |Lenox Hill East          

### 3.2 groupBy aggregations

* **By borough and hour:** number of trips, average fare, average tip percent, average trip length and total revenue.
* **By month:** trips and revenue across the year.
* **Complex aggregation, by route:** the most valuable pickup to drop-off zone pairs, keeping only routes with at least 5,000 trips.

In [9]:
borough_hour = (
    trips_enriched.groupBy("pickup_borough", "pickup_hour")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_pct"), 2).alias("avg_tip_pct"),
        F.round(F.avg("trip_minutes"), 1).alias("avg_minutes"),
        F.round(F.sum("total_amount"), 0).alias("total_revenue"),
    )
    .orderBy("pickup_borough", "pickup_hour")
)
borough_hour.filter(F.col("pickup_borough") == "Manhattan").show(24)

+--------------+-----------+-------+--------+-----------+-----------+-------------+
|pickup_borough|pickup_hour|  trips|avg_fare|avg_tip_pct|avg_minutes|total_revenue|
+--------------+-----------+-------+--------+-----------+-----------+-------------+
|     Manhattan|          0|2129904|   11.54|      19.59|       12.0|  3.6591882E7|
|     Manhattan|          1|1514643|   11.41|      19.81|       11.4|  2.5706985E7|
|     Manhattan|          2|1074721|   11.57|      19.69|       11.0|   1.835635E7|
|     Manhattan|          3| 746956|   12.35|      17.82|       11.2|  1.3395682E7|
|     Manhattan|          4| 552982|    14.8|      16.31|       12.1|  1.1582668E7|
|     Manhattan|          5| 616900|   13.95|      17.09|       10.7|    1.25893E7|
|     Manhattan|          6|1485279|   11.19|      17.99|        9.8|  2.4529786E7|
|     Manhattan|          7|2684219|   10.71|      19.26|       11.3|  4.2464255E7|
|     Manhattan|          8|3408269|   11.07|      19.22|       12.9|  5.530

In [10]:
monthly = (
    trips_enriched.groupBy("pickup_month")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("total_amount") / 1e6, 2).alias("revenue_millions"),
        F.round(F.avg("trip_distance"), 2).alias("avg_miles"),
        F.round(F.avg("avg_speed_mph"), 1).alias("avg_speed_mph"),
    )
    .orderBy("pickup_month")
)
monthly.show(12)

+------------+-------+----------------+---------+-------------+
|pickup_month|  trips|revenue_millions|avg_miles|avg_speed_mph|
+------------+-------+----------------+---------+-------------+
|           1|7441174|          115.42|     2.83|         11.7|
|           2|6800648|          125.95|     2.91|         11.4|
|           3|7582770|          144.01|     3.02|         11.4|
|           4|7192432|          137.57|     3.02|         11.4|
|           5|7308265|          142.21|     3.06|         11.2|
|           6|6694594|          131.22|     3.11|         11.2|
|           7|6041500|          117.05|     3.11|         11.6|
|           8|5809178|          112.92|     3.17|         11.9|
|           9|6289625|          123.94|     3.09|         11.1|
|          10|6903514|          134.89|     3.01|         10.8|
|          11|6574913|          126.16|     2.94|         10.9|
|          12|6589677|          127.54|     2.98|         11.0|
+------------+-------+----------------+-

In [11]:
top_routes = (
    trips_enriched.filter(F.col("pickup_zone") != F.col("dropoff_zone"))
    .groupBy("pickup_zone", "dropoff_zone")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("total_amount"), 2).alias("avg_total"),
        F.round(F.sum("total_amount"), 0).alias("revenue"),
    )
    .filter(F.col("trips") >= 5000)
    .orderBy(F.desc("revenue"))
    .limit(10)
)
top_routes.show(truncate=False)

+-------------------------+-------------------------+------+---------+---------+
|pickup_zone              |dropoff_zone             |trips |avg_total|revenue  |
+-------------------------+-------------------------+------+---------+---------+
|JFK Airport              |Times Sq/Theatre District|98215 |68.67    |6744500.0|
|LaGuardia Airport        |Times Sq/Theatre District|123967|52.05    |6452085.0|
|JFK Airport              |Outside of NYC           |66844 |94.17    |6294923.0|
|Upper East Side South    |Upper East Side North    |524770|11.23    |5894341.0|
|Upper East Side North    |Upper East Side South    |445990|11.75    |5242301.0|
|Times Sq/Theatre District|LaGuardia Airport        |89904 |51.86    |4662538.0|
|LaGuardia Airport        |Midtown Center           |88952 |50.03    |4450065.0|
|JFK Airport              |Clinton East             |63767 |68.14    |4345342.0|
|JFK Airport              |Midtown South            |60488 |69.94    |4230345.0|
|LaGuardia Airport        |M

## 4. SQL queries

We register the enriched DataFrame as a temporary view called `trips_enriched`. A view is just a name for the DataFrame, so SQL and the DataFrame code run through the same Spark optimizer.

* **Query 1:** tipping by borough on weekdays vs weekends (credit card trips only, because the data only records tips paid by card).
* **Query 2:** busiest hours of the day across the city.
* **Query 3:** the top 3 pickup zones in each borough, using a window function (`RANK() OVER (PARTITION BY ...)`).

In [12]:
trips_enriched.createOrReplaceTempView("trips_enriched")

sql_tipping = spark.sql("""
    SELECT pickup_borough,
           CASE WHEN is_weekend THEN 'Weekend' ELSE 'Weekday' END AS day_type,
           COUNT(*)                 AS trips,
           ROUND(AVG(tip_pct), 2)   AS avg_tip_pct,
           ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM trips_enriched
    WHERE payment_method = 'Credit card'
      AND pickup_borough IN ('Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island')
    GROUP BY pickup_borough, is_weekend
    ORDER BY pickup_borough, day_type
""")
sql_tipping.show(truncate=False)

26/11/20 01:59:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+--------+--------+-----------+--------+
|pickup_borough|day_type|trips   |avg_tip_pct|avg_fare|
+--------------+--------+--------+-----------+--------+
|Bronx         |Weekday |55449   |3.43       |26.8    |
|Bronx         |Weekend |19331   |4.66       |23.62   |
|Brooklyn      |Weekday |418266  |16.82      |18.18   |
|Brooklyn      |Weekend |187069  |18.46      |16.42   |
|Manhattan     |Weekday |40338508|25.76      |11.62   |
|Manhattan     |Weekend |13831249|25.75      |11.25   |
|Queens        |Weekday |2641167 |23.54      |39.35   |
|Queens        |Weekend |905406  |30.05      |39.09   |
|Staten Island |Weekday |1519    |16.6       |60.89   |
|Staten Island |Weekend |568     |7.55       |59.02   |
+--------------+--------+--------+-----------+--------+



In [13]:
sql_busiest_hours = spark.sql("""
    SELECT pickup_hour,
           COUNT(*)                     AS trips,
           ROUND(AVG(avg_speed_mph), 1) AS avg_speed_mph,
           ROUND(AVG(fare_amount), 2)   AS avg_fare
    FROM trips_enriched
    GROUP BY pickup_hour
    ORDER BY trips DESC
""")
sql_busiest_hours.show(24)

+-----------+-------+-------------+--------+
|pickup_hour|  trips|avg_speed_mph|avg_fare|
+-----------+-------+-------------+--------+
|         18|5345583|          9.9|    12.7|
|         19|5059093|         10.8|    12.4|
|         17|4832406|          9.6|   13.46|
|         20|4619546|         12.0|   12.78|
|         21|4576533|         12.5|   13.03|
|         15|4536453|          9.5|   13.86|
|         14|4516302|          9.7|   13.83|
|         13|4292102|         10.0|   13.39|
|         16|4262206|          9.7|   14.04|
|         22|4258994|         12.9|   13.17|
|         12|4253934|          9.8|   13.09|
|         11|4009983|          9.9|   12.79|
|         10|3821293|         10.3|   12.74|
|          9|3780043|         10.2|   12.56|
|          8|3702764|         10.5|   12.48|
|         23|3363955|         13.8|   13.55|
|          7|2946988|         12.5|   12.39|
|          0|2406993|         14.8|   13.68|
|          6|1672641|         16.0|    13.4|
|         

In [14]:
sql_top_zones = spark.sql("""
    WITH zone_counts AS (
        SELECT pickup_borough, pickup_zone, COUNT(*) AS trips
        FROM trips_enriched
        WHERE pickup_borough IS NOT NULL
          AND pickup_borough NOT IN ('Unknown', 'N/A')
        GROUP BY pickup_borough, pickup_zone
    ),
    ranked AS (
        SELECT *, RANK() OVER (PARTITION BY pickup_borough ORDER BY trips DESC) AS zone_rank
        FROM zone_counts
    )
    SELECT pickup_borough, zone_rank, pickup_zone, trips
    FROM ranked
    WHERE zone_rank <= 3
    ORDER BY pickup_borough, zone_rank
""")
sql_top_zones.show(30, truncate=False)

+--------------+---------+--------------------------------+-------+
|pickup_borough|zone_rank|pickup_zone                     |trips  |
+--------------+---------+--------------------------------+-------+
|Bronx         |1        |Mott Haven/Port Morris          |14447  |
|Bronx         |2        |West Concourse                  |11219  |
|Bronx         |3        |East Concourse/Concourse Village|6507   |
|Brooklyn      |1        |Downtown Brooklyn/MetroTech     |93466  |
|Brooklyn      |2        |Brooklyn Heights                |79004  |
|Brooklyn      |3        |Boerum Hill                     |65438  |
|EWR           |1        |Newark Airport                  |532    |
|Manhattan     |1        |Upper East Side South           |3542619|
|Manhattan     |2        |Midtown Center                  |3351261|
|Manhattan     |3        |Upper East Side North           |3201684|
|Queens        |1        |JFK Airport                     |2561736|
|Queens        |2        |LaGuardia Airport     

## 5. Optimization and execution plans

### 5.1 Physical plan of the main aggregation

`.explain(mode="formatted")` prints the **physical plan**: the steps Spark will actually run

In [15]:
borough_hour.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (21)
+- Sort (20)
   +- Exchange (19)
      +- HashAggregate (18)
         +- Exchange (17)
            +- HashAggregate (16)
               +- Project (15)
                  +- BroadcastHashJoin LeftOuter BuildRight (14)
                     :- Project (9)
                     :  +- BroadcastHashJoin LeftOuter BuildRight (8)
                     :     :- Project (3)
                     :     :  +- Filter (2)
                     :     :     +- Scan parquet  (1)
                     :     +- BroadcastExchange (7)
                     :        +- Project (6)
                     :           +- Filter (5)
                     :              +- Scan csv  (4)
                     +- BroadcastExchange (13)
                        +- Project (12)
                           +- Filter (11)
                              +- Scan csv  (10)


(1) Scan parquet 
Output [10]: [tpep_pickup_datetime#229, tpep_dropoff_datetime#230, passenger_count#231, trip_distanc

### 5.2 Partition pruning

The clean data is saved in one folder per month. So when we ask only for June, Spark opens just the June folder and skips the other 11. **Partition pruning** makes the query much faster.

In [16]:
june_only = trips.filter(F.col("file_month") == 6).select(
    "trip_distance", "fare_amount"
)
june_only.explain(mode="formatted")

all_files = glob.glob(os.path.join(CLEAN_DIR, "file_month=*", "*.parquet"))
june_files = glob.glob(os.path.join(CLEAN_DIR, "file_month=6", "*.parquet"))
print("Parquet files in the whole dataset:", len(all_files))
print("Parquet files Spark reads for June:", len(june_files))
timed("Count June trips (with partition pruning)", lambda: june_only.count())
timed(
    "Count June trips by filtering the timestamp instead (no pruning)",
    lambda: trips.filter(F.month("tpep_pickup_datetime") == 6).count(),
)

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [trip_distance#232, fare_amount#238, file_month#246]
Batched: true
Location: InMemoryFileIndex [file:/Users/mbaqir/spark-data/a12/clean_trips]
PartitionFilters: [isnotnull(file_month#246), (file_month#246 = 6)]
ReadSchema: struct<trip_distance:double,fare_amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [trip_distance#232, fare_amount#238, file_month#246]

(3) Project [codegen id : 1]
Output [2]: [trip_distance#232, fare_amount#238]
Input [3]: [trip_distance#232, fare_amount#238, file_month#246]


Parquet files in the whole dataset: 12
Parquet files Spark reads for June: 1
Count June trips (with partition pruning): 0.05 seconds
Count June trips by filtering the timestamp instead (no pruning): 0.73 seconds


(6971351, 0.7323342079980648)

### 5.3 Slow version vs fast version

We ask the same question twice: how many trips and what average fare per pickup borough.

| | Slow version | Fast version |
|---|---|---|
| Joining the zone table | Shuffle join: both tables are moved around and sorted | Broadcast join: the small zone table is copied to every worker, so the big table stays put |
| Pieces after regrouping | 200 (Spark default) | 16 (better for a laptop) |

The answers are the same. Only the speed changes.

In [ ]:
def borough_summary(df):
    return (
        df.groupBy("pickup_borough")
        .agg(
            F.count("*").alias("trips"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        )
        .orderBy(F.desc("trips"))
    )


# Naive: all columns, no broadcast, 200 shuffle partitions
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", "200")
naive = borough_summary(
    trips.join(pickup_zones, on="PULocationID", how="left").filter(
        F.col("fare_amount") > 0
    )
)
naive.explain()
naive_result, naive_seconds = timed("Naive version", lambda: naive.collect())

# Optimized: restore the settings, prune columns, broadcast the small table
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.set("spark.sql.shuffle.partitions", "16")
optimized = borough_summary(
    trips.select("PULocationID", "fare_amount")
    .filter(F.col("fare_amount") > 0)
    .join(F.broadcast(pickup_zones), on="PULocationID", how="left")
)
optimized.explain()
optimized_result, optimized_seconds = timed(
    "Optimized version", lambda: optimized.collect()
)

print(f"Speed-up: {naive_seconds / optimized_seconds:.1f}x faster")
optimized.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [trips#1066L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(trips#1066L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2398]
      +- HashAggregate(keys=[pickup_borough#378], functions=[count(1), avg(fare_amount#238)])
         +- Exchange hashpartitioning(pickup_borough#378, 200), ENSURE_REQUIREMENTS, [plan_id=2395]
            +- HashAggregate(keys=[pickup_borough#378], functions=[partial_count(1), partial_avg(fare_amount#238)])
               +- Project [fare_amount#238, pickup_borough#378]
                  +- SortMergeJoin [PULocationID#235L], [PULocationID#377L], LeftOuter
                     :- Sort [PULocationID#235L ASC NULLS FIRST], false, 0
                     :  +- Exchange hashpartitioning(PULocationID#235L, 200), ENSURE_REQUIREMENTS, [plan_id=2387]
                     :     +- Project [PULocationID#235L, fare_amount#238]
                     :        +- Filter (isnotnull(fare_amount#238) 

Naive version: 5.06 seconds
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [trips#1109L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(trips#1109L DESC NULLS LAST, 16), ENSURE_REQUIREMENTS, [plan_id=2679]
      +- HashAggregate(keys=[pickup_borough#378], functions=[count(1), avg(fare_amount#238)])
         +- Exchange hashpartitioning(pickup_borough#378, 16), ENSURE_REQUIREMENTS, [plan_id=2676]
            +- HashAggregate(keys=[pickup_borough#378], functions=[partial_count(1), partial_avg(fare_amount#238)])
               +- Project [fare_amount#238, pickup_borough#378]
                  +- BroadcastHashJoin [PULocationID#235L], [PULocationID#377L], LeftOuter, BuildRight, false, false
                     :- Project [PULocationID#235L, fare_amount#238]
                     :  +- Filter (isnotnull(fare_amount#238) AND (fare_amount#238 > 0.0))
                     :     +- FileScan parquet [PULocationID#235L,fare_amount#238,file_month#246] Batched: true, Dat

Optimized version: 0.86 seconds
Speed-up: 5.9x faster
+--------------+--------+--------+
|pickup_borough|   trips|avg_fare|
+--------------+--------+--------+
|     Manhattan|76455000|   11.53|
|        Queens| 5878948|    36.4|
|      Brooklyn| 1046584|   18.71|
|       Unknown|  779651|    16.0|
|         Bronx|  176307|   26.56|
|           N/A|   44743|   73.83|
|           EWR|    8319|   84.53|
| Staten Island|    3771|   53.89|
+--------------+--------+--------+



## 6. Caching

When the same DataFrame is used by several actions, Spark normally recomputes it from the Parquet files every time. `.cache()` keeps the computed rows in memory after the first action, so later actions reuse them.

To keep the cached data small enough for laptop memory, this demo caches the first quarter of 2019 (January to March) with only the columns the repeated queries need. We run the same three actions before and after caching.

In [18]:
q1_trips = trips_enriched.filter(F.col("file_month").between(1, 3)).select(
    "pickup_borough",
    "pickup_hour",
    "pickup_month",
    "fare_amount",
    "tip_pct",
    "trip_minutes",
)


def repeated_actions(df):
    df.count()
    df.groupBy("pickup_borough").agg(F.avg("fare_amount")).collect()
    df.groupBy("pickup_hour").agg(F.avg("tip_pct")).collect()


_, seconds_without_cache = timed(
    "3 actions WITHOUT cache", lambda: repeated_actions(q1_trips)
)

q1_trips.cache()
timed("First action after .cache() (fills the cache)", lambda: q1_trips.count())
_, seconds_with_cache = timed(
    "3 actions WITH cache", lambda: repeated_actions(q1_trips)
)

print(
    f"Caching made the repeated actions {seconds_without_cache / seconds_with_cache:.1f}x faster"
)
q1_trips.unpersist()

3 actions WITHOUT cache: 3.01 seconds


First action after .cache() (fills the cache): 3.36 seconds
3 actions WITH cache: 0.69 seconds
Caching made the repeated actions 4.4x faster


DataFrame[pickup_borough: string, pickup_hour: int, pickup_month: int, fare_amount: double, tip_pct: double, trip_minutes: double]

## 7. Actions vs transformations

* A **transformation** (`filter`, `select`, `withColumn`, `join`, `groupBy`) only describes work. Spark adds it to a plan and returns immediately. This is called **lazy evaluation**.
* An **action** (`count`, `show`, `collect`, `write`) needs a real answer, so Spark finally runs the whole plan as one or more **jobs**.

This is proved by timing both and by asking Spark how many jobs it started at each point.

In [19]:
sc = spark.sparkContext
tracker = sc.statusTracker()
sc.setJobGroup("lazy_demo", "Actions vs transformations demo")

start = time.perf_counter()
lazy_df = (
    trips.filter(F.col("trip_distance") > 10)
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount", "fare_per_mile")
)
transform_seconds = time.perf_counter() - start
jobs_after_transformations = len(tracker.getJobIdsForGroup("lazy_demo"))

start = time.perf_counter()
long_trip_count = lazy_df.count()
action_seconds = time.perf_counter() - start
jobs_after_action = len(tracker.getJobIdsForGroup("lazy_demo"))

sc.setLocalProperty("spark.jobGroup.id", None)

print(
    f"Transformations (filter, withColumn, select): {transform_seconds:.4f} seconds, Spark jobs started: {jobs_after_transformations}"
)
print(
    f"Action (count):                                {action_seconds:.4f} seconds, Spark jobs started: {jobs_after_action}"
)
print(f"Trips longer than 10 miles: {long_trip_count:,}")

Transformations (filter, withColumn, select): 0.0253 seconds, Spark jobs started: 0
Action (count):                                0.1499 seconds, Spark jobs started: 2
Trips longer than 10 miles: 5,319,752


## 8. Write results

* The full enriched trip table is written as Parquet, **partitioned by `pickup_month`**, to the data folder outside the repository (it is too large for GitHub).
* The small summary tables are written as Parquet next to it, and also saved as CSV files in the repository's `results/` folder so they can be viewed on GitHub.

In [20]:
timed(
    "Write enriched trips (partitioned by pickup_month)",
    lambda: trips_enriched.write.mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(os.path.join(OUTPUT_DIR, "trips_enriched")),
)

summaries = {
    "borough_hour_summary": borough_hour,
    "monthly_summary": monthly,
    "top_routes_by_revenue": top_routes,
    "sql_tipping_by_borough": sql_tipping,
    "sql_busiest_hours": sql_busiest_hours,
    "sql_top_zones_per_borough": sql_top_zones,
}

for name, df in summaries.items():
    df.coalesce(1).write.mode("overwrite").parquet(os.path.join(OUTPUT_DIR, name))
    df.toPandas().to_csv(os.path.join(RESULTS_DIR, f"{name}.csv"), index=False)
    print("Saved", name)

print("Parquet output folder:", OUTPUT_DIR)
print("CSV results folder:   ", RESULTS_DIR)

[42.063s][warning][gc,alloc] Executor task launch worker for task 5.0 in stage 125.0 (TID 383): Retried waiting for GCLocker too often allocating 2097152 words


26/11/20 01:59:37 WARN TaskMemoryManager: Failed to allocate a page (16777200 bytes), try spilling task memory.
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.unsafe.memory.HeapMemoryAllocator.allocate(HeapMemoryAllocator.java:72)
	at org.apache.spark.memory.TaskMemoryManager.allocatePageInternal(TaskMemoryManager.java:568)
	at org.apache.spark.memory.TaskMemoryManager.allocatePage(TaskMemoryManager.java:505)
	at org.apache.spark.memory.TaskMemoryManager.allocatePageWithMinimum(TaskMemoryManager.java:523)
	at org.apache.spark.memory.MemoryConsumer.allocatePage(MemoryConsumer.java:121)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.acquireNewPageIfNecessary(UnsafeExternalSorter.java:459)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.allocateMemoryForRecordIfNecessary(UnsafeExternalSorter.java:479)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.insertRecord(UnsafeExternalSorter.java:527)
	at org.apach

Write enriched trips (partitioned by pickup_month): 39.45 seconds


Saved borough_hour_summary


Saved monthly_summary


Saved top_routes_by_revenue


Saved sql_tipping_by_borough


Saved sql_busiest_hours


Saved sql_top_zones_per_borough
Parquet output folder: /Users/mbaqir/spark-data/a12/output
CSV results folder:    /Users/mbaqir/Library/CloudStorage/OneDrive-DukeUniversity/Semesters/Fall 2026/IDS 706 - Data Engineering/Assignment 12/results


## 9. Optional: MLlib linear regression

A simple regression that predicts `fare_amount` from trip distance, trip length in minutes, pickup hour, passenger count and a weekend flag. We use a 1% random sample so it trains quickly on a laptop.

In [21]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

FEATURES = [
    "trip_distance",
    "trip_minutes",
    "pickup_hour",
    "passenger_count",
    "weekend_flag",
]

ml_data = (
    trips_enriched.sample(fraction=0.01, seed=42)
    .withColumn("weekend_flag", F.col("is_weekend").cast("int"))
    .select(*FEATURES, "fare_amount")
    .dropna()
)
train, test = ml_data.randomSplit([0.8, 0.2], seed=42)

model = Pipeline(
    stages=[
        VectorAssembler(inputCols=FEATURES, outputCol="features"),
        LinearRegression(featuresCol="features", labelCol="fare_amount"),
    ]
).fit(train)

predictions = model.transform(test)
rmse = RegressionEvaluator(labelCol="fare_amount", metricName="rmse").evaluate(
    predictions
)
r2 = RegressionEvaluator(labelCol="fare_amount", metricName="r2").evaluate(predictions)

lr_model = model.stages[-1]
print(f"Training rows: {train.count():,}   Test rows: {test.count():,}")
print(f"Test RMSE: ${rmse:.2f}   Test R squared: {r2:.3f}")
print(f"Intercept: {lr_model.intercept:.3f}")
for name, coef in zip(FEATURES, lr_model.coefficients):
    print(f"  {name}: {coef:.3f}")

26/11/20 02:00:46 WARN Instrumentation: [363c31c0] regParam is zero, which might cause numerical instability and overfitting.


Training rows: 650,911   Test rows: 162,182
Test RMSE: $3.03   Test R squared: 0.931
Intercept: 2.687
  trip_distance: 2.096
  trip_minutes: 0.288
  pickup_hour: -0.004
  passenger_count: -0.010
  weekend_flag: -0.109


## Finish

The Spark UI stays available at the address printed in Section 1 (usually http://localhost:4040) until the session is stopped. Take the screenshots for the README **before** running the cell below.

In [23]:
spark.stop()